# Análisis Exploratorio de Datos - Modelo Comercial de Celulares

**Estudiante:** Anthony Isaac Mendoza Palomino  
**Módulo:** Friendly SQL & Python para Analytics  
**Fecha:** 2025  
**Dataset:** Modelo Comercial de Ventas de Celulares (dataset propio)  

---

## Objetivo
Desarrollar un análisis exploratorio completo sobre un dataset de ventas de celulares, identificando tendencias, patrones y hallazgos clave mediante Python, Plotly y consultas SQL con DuckDB.

El dataset contiene información de facturas de venta de celulares distribuidos en múltiples países, incluyendo datos de vendedores, clientes, tiendas, operadores y modelos de celulares.

## 1. Instalación e importación de librerías

In [ ]:
# Instalación de librerías necesarias
!pip install duckdb plotly openpyxl -q

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('Librerías importadas correctamente')

## 2. Carga del dataset

El dataset está compuesto por 8 hojas en un archivo Excel. La hoja principal es **Ventas**, que contiene las transacciones comerciales. Las demás hojas son tablas de referencia (dimensiones) que enriquecen el análisis.

In [ ]:
# Carga de todas las hojas del dataset
df_ventas = pd.read_excel('ModeloComercial.xlsx', sheet_name='Ventas')
df_pais = pd.read_excel('ModeloComercial.xlsx', sheet_name='País')
df_cliente = pd.read_excel('ModeloComercial.xlsx', sheet_name='Cliente')
df_vendedor = pd.read_excel('ModeloComercial.xlsx', sheet_name='Vendedor')
df_operador = pd.read_excel('ModeloComercial.xlsx', sheet_name='Operador')
df_marca = pd.read_excel('ModeloComercial.xlsx', sheet_name='Marca')
df_tienda = pd.read_excel('ModeloComercial.xlsx', sheet_name='Tienda')
df_modelo = pd.read_excel('ModeloComercial.xlsx', sheet_name='Modelo')

print('Dataset cargado correctamente')
print(f'Tabla principal Ventas: {df_ventas.shape[0]} filas y {df_ventas.shape[1]} columnas')
df_ventas.head()

## 3. Exploración inicial

Revisamos la estructura del dataset, los tipos de datos y las dimensiones de cada tabla.

In [ ]:
# Información general del dataset principal
print('=== INFORMACIÓN GENERAL - VENTAS ===')
print(f'Dimensiones: {df_ventas.shape}')
print(f'\nColumnas y tipos de datos:')
df_ventas.info()

In [ ]:
# Primeras y últimas filas
print('=== PRIMERAS 5 FILAS ===')
display(df_ventas.head())

print('\n=== ÚLTIMAS 5 FILAS ===')
display(df_ventas.tail())

In [ ]:
# Resumen de tablas de referencia
tablas = {
    'País': df_pais,
    'Cliente': df_cliente,
    'Vendedor': df_vendedor,
    'Operador': df_operador,
    'Marca': df_marca,
    'Tienda': df_tienda,
    'Modelo': df_modelo
}

print('=== RESUMEN DE TABLAS DE REFERENCIA ===')
for nombre, tabla in tablas.items():
    print(f'{nombre}: {tabla.shape[0]} registros | Columnas: {list(tabla.columns)}')

## 4. Análisis de calidad de datos

Identificamos valores nulos, duplicados e inconsistencias en el dataset.

In [ ]:
# Valores nulos
print('=== VALORES NULOS POR COLUMNA ===')
nulos = df_ventas.isnull().sum()
porcentaje = (nulos / len(df_ventas) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje %': porcentaje})
print(resumen_nulos)
print(f'\nTotal de valores nulos: {nulos.sum()}')

In [ ]:
# Duplicados
duplicados = df_ventas.duplicated().sum()
print(f'=== DUPLICADOS ===')
print(f'Total de filas duplicadas: {duplicados}')
print('No se encontraron duplicados en el dataset.' if duplicados == 0 else f'Se encontraron {duplicados} duplicados.')

In [ ]:
# Rango de fechas
print('=== RANGO DE FECHAS ===')
print(f'Fecha mínima de venta: {df_ventas["Fecha"].min().date()}')
print(f'Fecha máxima de venta: {df_ventas["Fecha"].max().date()}')
print(f'Período analizado: {(df_ventas["Fecha"].max() - df_ventas["Fecha"].min()).days} días')

## 5. Estadística descriptiva

Analizamos las variables numéricas y categóricas para entender la distribución de los datos.

In [ ]:
# Estadísticas descriptivas de variables numéricas
print('=== ESTADÍSTICAS DESCRIPTIVAS - VARIABLES NUMÉRICAS ===')
cols_numericas = ['Cantidad', 'Costo', 'Precio', 'Venta']
display(df_ventas[cols_numericas].describe().round(2))

In [ ]:
# Preparación del dataset enriquecido con dimensiones
df = df_ventas.copy()
df = df.merge(df_pais, on='CódigoPaís', how='left')
df = df.merge(df_vendedor, on='CódigoVendedor', how='left')
df = df.merge(df_marca.rename(columns={'CódigoMarca': 'CódigoMarca_temp'}), left_on='CódigoModelo', right_on='CódigoMarca_temp', how='left')
df = df.merge(df_tienda, on='CódigoTienda', how='left')
df = df.merge(df_operador, on='CódigoOperador', how='left')
df['Año'] = df['Fecha'].dt.year
df['Mes'] = df['Fecha'].dt.month
df['Margen'] = df['Venta'] - (df['Costo'] * df['Cantidad'])
df['Dias_Pago'] = (df['FechaPago'] - df['FechaVencimiento']).dt.days

print('Dataset enriquecido creado correctamente')
print(f'Dimensiones: {df.shape}')
df.head(3)

In [ ]:
# Resumen de variables categóricas
print('=== RESUMEN DE VARIABLES CATEGÓRICAS ===')
print(f'\nPaíses únicos: {df["NombrePaís"].nunique()}')
print(df['NombrePaís'].value_counts())

print(f'\nVendedores únicos: {df["NombreVendedor"].nunique()}')
print(df['NombreVendedor'].value_counts())

print(f'\nTiendas únicas: {df["RazónSocialTienda"].nunique()}')
print(df['RazónSocialTienda'].value_counts())

## 6. Visualizaciones con Plotly

Creamos visualizaciones interactivas para explorar los datos desde diferentes ángulos.

In [ ]:
# Gráfico 1: Evolución de ventas por año (líneas)
ventas_año = df.groupby('Año')['Venta'].sum().reset_index()

fig1 = px.line(
    ventas_año,
    x='Año',
    y='Venta',
    title='Evolución del total de ventas por año (2014-2020)',
    markers=True,
    labels={'Venta': 'Total ventas ($)', 'Año': 'Año'},
    color_discrete_sequence=['#0F6E56']
)
fig1.update_traces(line_width=3, marker_size=8)
fig1.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig1.show()

print('Interpretación: Las ventas crecieron hasta 2017 (pico máximo de $6.9 mill.) y luego cayeron sostenidamente hasta 2020, su punto más bajo con $3.2 mill.')

In [ ]:
# Gráfico 2: Total de ventas por vendedor (barras)
ventas_vendedor = df.groupby('NombreVendedor')['Venta'].sum().reset_index().sort_values('Venta', ascending=True)

fig2 = px.bar(
    ventas_vendedor,
    x='Venta',
    y='NombreVendedor',
    orientation='h',
    title='Total de ventas por vendedor',
    labels={'Venta': 'Total ventas ($)', 'NombreVendedor': 'Vendedor'},
    color='Venta',
    color_continuous_scale='Greens'
)
fig2.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig2.show()

print('Interpretación: María lidera las ventas con $13.4 mill., seguida por Tomás con $9.6 mill. Existe una brecha significativa entre los dos primeros vendedores y el resto.')

In [ ]:
# Gráfico 3: Distribución de ventas por país (pastel)
ventas_pais = df.groupby('NombrePaís')['Venta'].sum().reset_index().sort_values('Venta', ascending=False)

fig3 = px.pie(
    ventas_pais,
    names='NombrePaís',
    values='Venta',
    title='Distribución de ventas por país',
    hole=0.3
)
fig3.update_layout(paper_bgcolor='white')
fig3.show()

print('Interpretación: Las ventas están distribuidas entre múltiples países. Se puede identificar cuáles mercados concentran mayor participación del negocio.')

In [ ]:
# Gráfico 4: Boxplot de distribución del monto de venta
fig4 = px.box(
    df,
    y='Venta',
    x='NombreVendedor',
    title='Distribución del monto de venta por vendedor',
    labels={'Venta': 'Monto de venta ($)', 'NombreVendedor': 'Vendedor'},
    color='NombreVendedor',
    color_discrete_sequence=px.colors.sequential.Greens
)
fig4.update_layout(plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig4.show()

print('Interpretación: El boxplot muestra la dispersión de los montos de venta individuales por vendedor, permitiendo identificar valores atípicos y diferencias en el ticket promedio de cada uno.')

In [ ]:
# Gráfico 5: Histograma de distribución del monto de venta
fig5 = px.histogram(
    df,
    x='Venta',
    nbins=30,
    title='Distribución del monto de venta por factura',
    labels={'Venta': 'Monto de venta ($)', 'count': 'Frecuencia'},
    color_discrete_sequence=['#0F6E56']
)
fig5.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig5.show()

print('Interpretación: La mayoría de las facturas se concentran en rangos de venta bajos a medios. Se observan pocas facturas de montos muy elevados, lo que indica que las ventas grandes son poco frecuentes.')

In [ ]:
# Gráfico 6: Scatter plot Costo vs Venta
fig6 = px.scatter(
    df,
    x='Costo',
    y='Venta',
    color='NombreVendedor',
    title='Relación entre costo y venta por vendedor',
    labels={'Costo': 'Costo ($)', 'Venta': 'Venta ($)'},
    opacity=0.7
)
fig6.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig6.show()

print('Interpretación: Se observa una relación positiva entre el costo y el monto de venta, lo que es esperado. Los puntos alejados de la tendencia central podrían indicar ventas con márgenes atípicos.')

## 7. Consultas SQL con DuckDB

Aplicamos consultas SQL directamente sobre los DataFrames de pandas usando DuckDB.

In [ ]:
# Consulta 1: Inspección inicial de datos (SELECT con LIMIT)
consulta1 = duckdb.query("""
    SELECT 
        Factura,
        Fecha,
        CódigoPaís,
        Cantidad,
        Costo,
        Precio,
        Venta
    FROM df_ventas
    ORDER BY Fecha DESC
    LIMIT 10
""").to_df()

print('=== CONSULTA 1: Últimas 10 facturas registradas ===')
display(consulta1)

In [ ]:
# Consulta 2: Ventas por año (GROUP BY con agregación)
consulta2 = duckdb.query("""
    SELECT 
        YEAR(Fecha) AS Año,
        COUNT(*) AS Total_Facturas,
        SUM(Venta) AS Total_Ventas,
        ROUND(AVG(Venta), 2) AS Venta_Promedio,
        SUM(Cantidad) AS Total_Unidades
    FROM df_ventas
    GROUP BY YEAR(Fecha)
    ORDER BY Año
""").to_df()

print('=== CONSULTA 2: Resumen de ventas por año ===')
display(consulta2)

In [ ]:
# Consulta 3: Filtrar ventas mayores al promedio (WHERE)
consulta3 = duckdb.query("""
    SELECT 
        Factura,
        Fecha,
        CódigoPaís,
        Venta
    FROM df_ventas
    WHERE Venta > (SELECT AVG(Venta) FROM df_ventas)
    ORDER BY Venta DESC
    LIMIT 15
""").to_df()

print('=== CONSULTA 3: Facturas con venta superior al promedio ===')
print(f'Venta promedio: ${df_ventas["Venta"].mean():.2f}')
display(consulta3)

In [ ]:
# Consulta 4: Ranking de países por ventas totales (ORDER BY)
consulta4 = duckdb.query("""
    SELECT 
        v.CódigoPaís,
        p.NombrePaís,
        COUNT(*) AS Total_Facturas,
        SUM(v.Venta) AS Total_Ventas,
        ROUND(AVG(v.Venta), 2) AS Venta_Promedio
    FROM df_ventas v
    JOIN df_pais p ON v.CódigoPaís = p.CódigoPaís
    GROUP BY v.CódigoPaís, p.NombrePaís
    ORDER BY Total_Ventas DESC
""").to_df()

print('=== CONSULTA 4: Ranking de países por total de ventas ===')
display(consulta4)

In [ ]:
# Consulta 5: Clientes con pagos atrasados (combinación de condiciones y cálculos)
consulta5 = duckdb.query("""
    SELECT 
        v.Factura,
        v.Fecha,
        c.RazónSocial AS Cliente,
        v.Venta,
        v.FechaVencimiento,
        v.FechaPago,
        DATEDIFF('day', v.FechaVencimiento, v.FechaPago) AS Dias_Atraso
    FROM df_ventas v
    JOIN df_cliente c ON v.CódigoCliente = c.CódigoCliente
    WHERE v.FechaPago > v.FechaVencimiento
    ORDER BY Dias_Atraso DESC
    LIMIT 15
""").to_df()

print('=== CONSULTA 5: Clientes con pagos atrasados ===')
print(f'Total de facturas con atraso: {len(consulta5)}')
display(consulta5)

In [ ]:
# Consulta 6: Margen bruto por vendedor
consulta6 = duckdb.query("""
    SELECT 
        vd.NombreVendedor,
        COUNT(*) AS Total_Facturas,
        SUM(v.Venta) AS Total_Ventas,
        SUM(v.Costo * v.Cantidad) AS Total_Costos,
        SUM(v.Venta) - SUM(v.Costo * v.Cantidad) AS Margen_Bruto,
        ROUND((SUM(v.Venta) - SUM(v.Costo * v.Cantidad)) / SUM(v.Venta) * 100, 2) AS Margen_Pct
    FROM df_ventas v
    JOIN df_vendedor vd ON v.CódigoVendedor = vd.CódigoVendedor
    GROUP BY vd.NombreVendedor
    ORDER BY Margen_Bruto DESC
""").to_df()

print('=== CONSULTA 6: Margen bruto por vendedor ===')
display(consulta6)

## 8. Hallazgos y conclusiones

A partir del análisis exploratorio realizado, se identificaron los siguientes hallazgos principales:

### Tendencia de ventas
Las ventas de la empresa mostraron un crecimiento sostenido desde 2014 hasta alcanzar su pico máximo en 2017 con $6.9 millones. A partir de ese año se produjo una caída progresiva que llegó a su punto más bajo en 2020 con $3.2 millones, representando una disminución del 54% respecto al pico máximo.

### Desempeño de vendedores
María es la vendedora con mayor volumen de ventas, concentrando aproximadamente el 35% del total. Existe una brecha significativa entre los dos primeros vendedores (María y Tomás) y el resto del equipo, lo que sugiere una oportunidad de mejora en la productividad de los demás vendedores.

### Distribución geográfica
Las ventas están distribuidas en múltiples países, lo que permite a la empresa diversificar el riesgo comercial. Sin embargo, algunos mercados concentran mayor participación que otros.

### Calidad de la cobranza
Se identificaron facturas con pagos realizados después de la fecha de vencimiento, lo que representa un riesgo para el flujo de caja de la empresa. Se recomienda implementar un seguimiento más estricto de los clientes con histórico de atrasos.

### Margen bruto
El margen bruto general del negocio es positivo pero relativamente bajo, lo que indica que la empresa opera con márgenes ajustados. Se recomienda analizar la estructura de costos por modelo de celular para identificar oportunidades de mejora en la rentabilidad.